# RAG Playground (Reranking Branch)

Dieses Notebook ist fuer den `reranking`-Branch vorbereitet.

Es deckt drei Teststufen ab:
- Retrieval-Vergleich fuer `bm25`, `dense` und `hybrid`
- optionales manuelles Reranking auf bereits geholten Dokumenten
- optionaler End-to-End-RAG-Lauf mit oder ohne Reranker


In [ ]:
# Setup: Projekt-Root robust setzen
import os
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

print('cwd:', cwd)
print('ROOT:', ROOT)
print('Working dir:', Path.cwd())
print('Python:', sys.executable)


In [ ]:
# Imports + Projektstatus
import json
import time
from statistics import mean

from src.config import RAW_DIR, CHROMA_DIR, DEFAULT_LLM_MODEL
from src.indexing import build_index, load_index, clear_retrieval_caches
from src.rag import get_docs_for_query, answer_with_rag_mode
from src.reranking import rerank_documents, DEFAULT_RERANKER_MODEL

pdfs = sorted(RAW_DIR.rglob('*.pdf')) if RAW_DIR.exists() else []
print('RAW_DIR:', RAW_DIR)
print('CHROMA_DIR:', CHROMA_DIR)
print('PDF count:', len(pdfs))
for p in pdfs[:10]:
    print('-', p.name)
if len(pdfs) > 10:
    print(f'... ({len(pdfs) - 10} more)')
print('Index dir exists:', CHROMA_DIR.exists())
print('Default LLM model:', DEFAULT_LLM_MODEL)
print('Default reranker model:', DEFAULT_RERANKER_MODEL)


In [ ]:
# Konfiguration fuer Tests
QUESTION_ID = 'q1'
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

REBUILD_INDEX = False
FORCE_REBUILD_INDEX = False

RETRIEVAL_K = 10
RETRIEVAL_RUNS = 2
RETRIEVAL_MODES = ('bm25', 'dense', 'hybrid')

RUN_MANUAL_RERANK = True
MANUAL_RERANK_MODE = 'hybrid'
MANUAL_RERANK_K = 20
MANUAL_RERANK_TOP_N = 5
MANUAL_RERANK_USE_FP16 = False

RUN_RAG = True
RAG_MODE = 'hybrid'
RAG_K = 20
USE_RERANKER = True
RERANK_TOP_N = 5
RERANKER_USE_FP16 = False

print('QUESTION_ID:', QUESTION_ID)
print('Chunking:', CHUNK_SIZE, CHUNK_OVERLAP)
print('Rebuild index:', REBUILD_INDEX, 'force:', FORCE_REBUILD_INDEX)
print('Retrieval test:', RETRIEVAL_MODES, 'k=', RETRIEVAL_K, 'runs=', RETRIEVAL_RUNS)
print('Manual rerank:', RUN_MANUAL_RERANK, MANUAL_RERANK_MODE, MANUAL_RERANK_K, MANUAL_RERANK_TOP_N)
print('RAG:', RUN_RAG, RAG_MODE, 'k=', RAG_K, 'reranker=', USE_RERANKER, 'top_n=', RERANK_TOP_N)


In [ ]:
# Fragen zentral laden
QUESTIONS_PATH = ROOT / 'data' / 'eval' / 'questions.jsonl'

def load_questions(path=QUESTIONS_PATH):
    items = {}
    with Path(path).open('r', encoding='utf-8-sig') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            qid = str(obj['id']).strip().lstrip('\ufeff')
            items[qid] = obj['query']
    return items

questions = load_questions()
print('Questions file:', QUESTIONS_PATH)
print('Verfuegbare IDs:', ', '.join(sorted(questions.keys())))

query = questions[QUESTION_ID]
print('Ausgewaehlte ID:', QUESTION_ID)
print('Frage:', query)


In [ ]:
# Index laden oder bei Bedarf neu bauen
if REBUILD_INDEX:
    print('Rebuilding index ...')
    vs = build_index(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        force_rebuild=FORCE_REBUILD_INDEX,
    )
    print('Index rebuilt.')
else:
    vs = load_index()
    print('Existing index loaded.')

# Optional nach Code-/Datenaenderungen:
# clear_retrieval_caches()


In [ ]:
# Helpers: Retrieval-Benchmark + Anzeige
def _preview(text: str, n: int = 240) -> str:
    text = ' '.join((text or '').split())
    return text[:n] + ('...' if len(text) > n else '')

def show_docs(docs, title='Docs'):
    print(f'\n=== {title} | count={len(docs)} ===')
    for i, d in enumerate(docs, start=1):
        md = d.metadata or {}
        print(f"{i}. source={md.get('source')}, page={md.get('page')}, section={md.get('section', '?')}")
        print('   ' + _preview(d.page_content))

def benchmark_retrieval(query: str, modes=('bm25', 'dense', 'hybrid'), k: int = 5, runs: int = 3, chunk_size: int = 1200, chunk_overlap: int = 200):
    print(f'Query: {query}')
    print(f'Modes={modes}, k={k}, runs={runs}, chunk_size={chunk_size}, chunk_overlap={chunk_overlap}')
    print('-' * 90)
    results = {}

    for mode in modes:
        latencies = []
        last_docs = []
        for idx in range(runs):
            t0 = time.perf_counter()
            docs = get_docs_for_query(
                query=query,
                mode=mode,
                k=k,
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
            )
            dt = time.perf_counter() - t0
            latencies.append(dt)
            last_docs = docs
            print(f'[{mode}] run {idx + 1}/{runs}: {dt:.3f}s | docs={len(docs)}')

        avg_s = mean(latencies) if latencies else float('nan')
        results[mode] = {'latencies': latencies, 'avg_s': avg_s, 'docs': last_docs}
        print(f'[{mode}] avg: {avg_s:.3f}s')
        print('-' * 90)

    print('\nTop-Chunks aus letztem Lauf je Modus:')
    for mode in modes:
        show_docs(results[mode]['docs'], title=mode.upper())

    return results


In [ ]:
# Retrieval-Vergleich
retrieval_results = benchmark_retrieval(
    query=query,
    modes=RETRIEVAL_MODES,
    k=RETRIEVAL_K,
    runs=RETRIEVAL_RUNS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)


In [ ]:
# Optional: manuelles Reranking auf Retrieval-Kandidaten
if RUN_MANUAL_RERANK:
    docs = get_docs_for_query(
        query=query,
        mode=MANUAL_RERANK_MODE,
        k=MANUAL_RERANK_K,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    print(f'Vor Reranking: {len(docs)} Dokumente')
    show_docs(docs[:min(5, len(docs))], title=f'{MANUAL_RERANK_MODE.upper()} BEFORE')

    reranked_docs = rerank_documents(
        query=query,
        docs=docs,
        top_n=MANUAL_RERANK_TOP_N,
        use_fp16=MANUAL_RERANK_USE_FP16,
    )
    show_docs(reranked_docs, title='RERANKED')
else:
    print('RUN_MANUAL_RERANK=False -> diese Stufe wird uebersprungen.')


In [ ]:
# Optional: End-to-End RAG-Lauf
if RUN_RAG:
    t0 = time.perf_counter()
    rag_res = answer_with_rag_mode(
        query=query,
        mode=RAG_MODE,
        k=RAG_K,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        use_reranker=USE_RERANKER,
        rerank_top_n=RERANK_TOP_N,
        reranker_use_fp16=RERANKER_USE_FP16,
    )
    dt = time.perf_counter() - t0

    print(f'RAG mode={RAG_MODE} total={dt:.3f}s | reranker={USE_RERANKER}')
    print('\nAntwort:\n', rag_res['answer'])
    print('\nQuellen:')
    for s in rag_res.get('sources', []):
        print(f"- {s.get('source')} (Seite {s.get('page')})")
else:
    print('RUN_RAG=False -> nur Retrieval ausgefuehrt.')
